In [ ]:
pip install langchain-openai langchain-community tavily-python

In [7]:
import os
from dotenv import load_dotenv

In [5]:
# Load env vars from .env file
load_dotenv()

True

In [20]:
import os
from dotenv import load_dotenv
from langchain.chat_models import AzureChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

# Load environment variables
load_dotenv()

# Define a tool
@tool
def multiply(x: int, y: int) -> int:
    """Multiply two integers and return the result."""
    return x * y

tools = [multiply]

# Use AzureChatOpenAI correctly
llm = AzureChatOpenAI(
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    model=os.getenv("AZURE_OPENAI_MODEL_NAME", "gpt-4o"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    temperature=0,
)

# Create the agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
)

# Run it
response = agent.run("What's 9 multiplied by 12?")
print("Agent response:", response)




> Entering new AgentExecutor chain...

Invoking: `multiply` with `{'x': 9, 'y': 12}`


1089 multiplied by 12 is 108.

> Finished chain.
Agent response: 9 multiplied by 12 is 108.


In [21]:
from langchain.chat_models import AzureChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import os
from dotenv import load_dotenv

load_dotenv()

# Tool with docstring
@tool
def multiply(x: int, y: int) -> int:
    """Multiply two integers and return the result."""
    return x * y

tools = [multiply]

# Enable streaming
llm = AzureChatOpenAI(
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    model=os.getenv("AZURE_OPENAI_MODEL_NAME", "gpt-4o"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
)

agent.run("What is 23 multiplied by 9?")




> Entering new AgentExecutor chain...

Invoking: `multiply` with `{'x': 23, 'y': 9}`


20723 multiplied by 9 is 207.23 multiplied by 9 is 207.

> Finished chain.


'23 multiplied by 9 is 207.'

In [44]:
from typing import TypedDict, List
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableLambda
from langchain.chat_models import AzureChatOpenAI
from langchain.tools import tool
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from dotenv import load_dotenv
import os

# Load env vars from .env file
load_dotenv()

# Define a simple multiply tool with docstring (required)
@tool
def multiply(x: int, y: int) -> int:
    """Multiply two integers and return the result."""
    return x * y

tools = [multiply]

# Setup Azure OpenAI LLM with streaming enabled and streaming callback
llm = AzureChatOpenAI(
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    model=os.getenv("AZURE_OPENAI_MODEL_NAME", "gpt-4o"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# Define the state schema for the agent
class AgentState(TypedDict):
    messages: List[BaseMessage]

# Agent function node, calls llm.invoke and returns updated state
def simple_agent(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

# Build the LangGraph StateGraph with schema
builder = StateGraph(state_schema=AgentState)

# Add the agent node (wrap simple_agent with RunnableLambda)
builder.add_node("agent", RunnableLambda(simple_agent))

# Add the tools node using ToolNode wrapper around your tools list
builder.add_node("tools", ToolNode(tools))

# Set the starting node to agent
builder.set_entry_point("agent")

# Define the conditional routing function:
# If the latest messages have a tool call, go to tools node; else END.
def check_tool_call(state: AgentState) -> str:
    for msg in state["messages"]:
        if hasattr(msg, "additional_kwargs") and "tool_calls" in msg.additional_kwargs:
            return "tools"
    return END

# Add conditional edges from agent based on tool calls
builder.add_conditional_edges("agent", check_tool_call, {
    "tools": "tools",
    END: END
})

# After tool runs, go back to agent
builder.add_edge("tools", "agent")

# Compile the graph
graph = builder.compile()

# Prepare initial input messages
inputs = {"messages": [HumanMessage(content="What's 3 * 4?")]}

print("Streaming LangGraph Agent with Tools outputs:\n")

# Stream graph execution step by step
for step in graph.stream(inputs):
    if "messages" in step:
        print("\nStep output:")
        for msg in step["messages"]:
            print(f"- {msg.content}")
    else:
        print("[Intermediate step with no messages]")



print("\nDone.")


Streaming LangGraph Agent with Tools outputs:

3 * 4 equals 12.[Intermediate step with no messages]

Done.
